In [0]:
%sql
-- 1. Silver Schema Creation
CREATE SCHEMA IF NOT EXISTS workspace.silver;

-- 2. Data Quality Audit / Quarantine Table
CREATE OR REPLACE TABLE workspace.silver.sales_order_quarantine (
    QuarantineKey BIGINT GENERATED ALWAYS AS IDENTITY,
    SourceEntity STRING NOT NULL,         -- 'HEADER' or 'ITEM'
    RecordIdentifier STRING,              -- SalesOrder or SalesOrder + SalesOrderItem
    FailedRule STRING NOT NULL,           -- e.g. 'RULE_1_NULL_PK', 'RULE_8_QUADRATURE_MISMATCH'
    FailureSeverity STRING NOT NULL,     -- 'HARD' or 'SOFT'
    RawRecord STRING NOT NULL,            -- Raw snapshot of the JSON payload
    IngestionTimestamp TIMESTAMP NOT NULL
)
CLUSTER BY (SourceEntity, FailedRule);

%sql
-- 3. Date Dimension (Calendar)
CREATE OR REPLACE TABLE workspace.silver.dim_date (
    DateKey INT NOT NULL,                 -- YYYYMMDD format
    FullDate DATE NOT NULL,
    YearNbr INT NOT NULL,
    QuarterNbr INT NOT NULL,
    MonthNbr INT NOT NULL,
    MonthName STRING NOT NULL,
    DayNbr INT NOT NULL,
    DayOfWeekName STRING NOT NULL,
    CONSTRAINT pk_dim_date PRIMARY KEY (DateKey)
)
CLUSTER BY (DateKey);

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.dim_customer_tmp (
    -- Real SAP Commercial Attributes (Bronze Header)
    SoldToParty STRING,
    CustomerGroup STRING,
    CustomerAccountAssignmentGroup STRING,
    CustomerPaymentTerms STRING,
    
    -- Metadata and Audit
    _rescued_data STRING,
    _ingestion_timestamp TIMESTAMP,

    -- State Machine Flags (Status Board)
    Status_Cleansing STRING,  -- 'PENDING', 'COMPLETED'
    Status_DQ1 STRING,        -- Rule 1: Mandatory fields (SoldToParty not null/empty)
    Status_DQ2 STRING,        -- Rule 2: Uniqueness (unique SoldToParty per batch)
    Status_DQ3 STRING,        -- Rule 3: ALPHA conversion (no leading zeros)
    Status_DQ4 STRING,        -- Rule 4: Master fields domain and length validation
    Status_DQ5 STRING,        -- Rule 5: Clean format (_rescued_data IS NULL)
    
    -- Lifecycle Control
    Status_Process STRING     -- 'IN_PROGRESS', 'QUARANTINED', 'READY_FOR_STG'
)
USING DELTA;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.dim_customer_stg (
    CustomerKey BIGINT GENERATED ALWAYS AS IDENTITY,
    SoldToParty STRING NOT NULL,
    CustomerGroup STRING,
    CustomerAccountAssignmentGroup STRING,
    CustomerPaymentTerms STRING,
    
    -- SCD Type 2 History Control
    ValidFrom TIMESTAMP NOT NULL,
    ValidTo TIMESTAMP,
    IsCurrent BOOLEAN NOT NULL,
    
    CONSTRAINT pk_dim_customer_stg PRIMARY KEY (CustomerKey)
)
CLUSTER BY (SoldToParty);

In [0]:
%sql
-- Idempotent insertion of sentinel record (Unknown)
MERGE INTO workspace.silver.dim_customer_stg AS tgt
USING (
    SELECT 
        'UNKNOWN' AS SoldToParty,
        'N/A' AS CustomerGroup,
        'N/A' AS CustomerAccountAssignmentGroup,
        'N/A' AS CustomerPaymentTerms,
        TIMESTAMP('1900-01-01 00:00:00') AS ValidFrom,
        CAST(NULL AS TIMESTAMP) AS ValidTo,
        TRUE AS IsCurrent
) AS src
ON tgt.SoldToParty = src.SoldToParty
WHEN NOT MATCHED THEN
    INSERT (
        SoldToParty,
        CustomerGroup,
        CustomerAccountAssignmentGroup,
        CustomerPaymentTerms,
        ValidFrom,
        ValidTo,
        IsCurrent
    )
    VALUES (
        src.SoldToParty,
        src.CustomerGroup,
        src.CustomerAccountAssignmentGroup,
        src.CustomerPaymentTerms,
        src.ValidFrom,
        src.ValidTo,
        src.IsCurrent
    );

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.dim_product_tmp (
    -- Product Attributes (Bronze Item)
    Material STRING,
    MaterialByCustomer STRING,
    MaterialGroup STRING,
    MaterialPricingGroup STRING,
    OriginallyRequestedMaterial STRING,
    
    -- Metadata and Audit
    _rescued_data STRING,
    _ingestion_timestamp TIMESTAMP,

    -- State Machine Flags (Status Board)
    Status_Cleansing STRING,  -- 'PENDING', 'COMPLETED'
    Status_DQ1 STRING,        -- Rule 1: Mandatory fields(Material NOT NULL/EMPTY)
    Status_DQ2 STRING,        -- Rule 2: Uniqueness/Consistency (unique Material per batch)
    Status_DQ3 STRING,        -- Rule 3: ALPHA conversion (no leading zeros)
    Status_DQ4 STRING,        -- Rule 4: Master fields domain and length validation (MaterialGroup <= 9, etc.)
    Status_DQ5 STRING,        -- Rule 5: Clean format(_rescued_data IS NULL)
    
    -- Lifecycle Control
    Status_Process STRING     -- 'IN_PROGRESS', 'QUARANTINED', 'READY_FOR_STG'
)
USING DELTA;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.dim_product_stg (
    ProductKey BIGINT GENERATED ALWAYS AS IDENTITY,
    Material STRING NOT NULL,
    MaterialByCustomer STRING,
    MaterialGroup STRING,
    MaterialPricingGroup STRING,
    OriginallyRequestedMaterial STRING,
    
    -- SCD Type 2 History Control
    ValidFrom TIMESTAMP NOT NULL,
    ValidTo TIMESTAMP,
    IsCurrent BOOLEAN NOT NULL,
    
    CONSTRAINT pk_dim_product_stg PRIMARY KEY (ProductKey)
)
CLUSTER BY (Material);

In [0]:
%sql
-- Idempotent insertion of sentinel record (Unknown)
MERGE INTO workspace.silver.dim_product_stg AS tgt
USING (
    SELECT 
        'UNKNOWN' AS Material,
        'N/A' AS MaterialByCustomer,
        'N/A' AS MaterialGroup,
        'N/A' AS MaterialPricingGroup,
        'N/A' AS OriginallyRequestedMaterial,
        TIMESTAMP('1900-01-01 00:00:00') AS ValidFrom,
        CAST(NULL AS TIMESTAMP) AS ValidTo,
        TRUE AS IsCurrent
) AS src
ON tgt.Material = src.Material
WHEN NOT MATCHED THEN
    INSERT (
        Material,
        MaterialByCustomer,
        MaterialGroup,
        MaterialPricingGroup,
        OriginallyRequestedMaterial,
        ValidFrom,
        ValidTo,
        IsCurrent
    )
    VALUES (
        src.Material,
        src.MaterialByCustomer,
        src.MaterialGroup,
        src.MaterialPricingGroup,
        src.OriginallyRequestedMaterial,
        src.ValidFrom,
        src.ValidTo,
        src.IsCurrent
    );

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.dim_organization_tmp (
    SalesOrganization STRING,
    DistributionChannel STRING,
    OrganizationDivision STRING,
    
    -- Technical Metadata & Audit
    _rescued_data STRING,
    _ingestion_timestamp TIMESTAMP,
    
    -- State Machine Flags (Status Board)
    Status_Cleansing STRING,
    Status_DQ1 STRING,         -- Non-null and non-empty keys
    Status_DQ2 STRING,         -- SAP standard field lengths (VKORG: 4, VTWEG: 2, SPART: 2)
    Status_DQ3 STRING,         -- Technical integrity (_rescued_data IS NULL)
    Status_Process STRING      -- 'IN_PROGRESS', 'QUARANTINED', 'READY_FOR_STG'
)
USING DELTA;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.dim_organization_stg (
    OrganizationKey BIGINT GENERATED ALWAYS AS IDENTITY,
    
    -- Composite Natural Key (SAP Sales Area)
    SalesOrganization STRING NOT NULL,
    DistributionChannel STRING NOT NULL,
    OrganizationDivision STRING NOT NULL,
    
    -- SCD Type 2 History Control Columns
    ValidFrom TIMESTAMP NOT NULL,
    ValidTo TIMESTAMP,
    IsCurrent BOOLEAN NOT NULL,
    
    -- Relational Integrity
    CONSTRAINT pk_dim_organization PRIMARY KEY (OrganizationKey)
)
CLUSTER BY (SalesOrganization);

In [0]:
%sql
-- Idempotent insertion of sentinel record (Unknown)
INSERT INTO workspace.silver.dim_organization_stg (
    SalesOrganization,
    DistributionChannel,
    OrganizationDivision,
    ValidFrom,
    ValidTo,
    IsCurrent
) VALUES (
    'UNKNOWN',
    'NA',
    'NA',
    TIMESTAMP('1900-01-01 00:00:00'),
    NULL,
    TRUE
);

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.fact_sales_order_item_tmp (
    -- Degenerate Keys
    SalesOrder STRING,
    SalesOrderItem STRING,
    
    -- Dimension Natural Keys (for lookup)
    SoldToParty STRING,
    Material STRING,
    
    -- Derived Surrogate Keys
    CustomerKey BIGINT,
    ProductKey BIGINT,
    OrganizationKey BIGINT,
    CreationDateKey INT,
    SalesOrderDateKey INT,
    RequestedDeliveryDateKey INT,
    
    -- Raw date formats for validation
    CreationDate_Raw STRING,
    SalesOrderDate_Raw STRING,
    RequestedDeliveryDate_Raw STRING,
    
    -- Status and Currency Fields
    SalesOrderItemCategory STRING,
    DeliveryStatus STRING,
    SDProcessStatus STRING,
    BillingStatus STRING,
    TransactionCurrency STRING,
    
    -- Item Metrics
    RequestedQuantity DECIMAL(18, 3),
    ConfdDeliveredQuantity DECIMAL(18, 3),
    NetAmount DECIMAL(18, 2),
    TaxAmount DECIMAL(18, 2),
    CostAmount DECIMAL(18, 2),
    GrossWeight DECIMAL(18, 3),
    NetWeight DECIMAL(18, 3),
    ItemVolume DECIMAL(18, 3),
    
    -- Header Metric (for reconciliation)
    HeaderTotalNetAmount DECIMAL(18, 2),
    
    -- Technical Metadata & Audit
    _rescued_data STRING,
    _ingestion_timestamp TIMESTAMP,
    
    -- Soft Data Quality (Warnings)
    DqWarnings ARRAY<STRING>,
    
    -- State Machine Flags (Hard DQ Rules)
    Status_Cleansing STRING,
    Status_DQ_Keys STRING,         -- Non-null Primary Keys
    Status_DQ_Format STRING,       -- Parseable dates and ISO currency
    Status_DQ_Metrics STRING,      -- Quantity > 0, valid amounts
    Status_DQ_Dates STRING,        -- Temporal consistency (Delivery >= Creation)
    Status_DQ_Integrity STRING,    -- _rescued_data IS NULL
    
    -- Lifecycle Control
    Status_Process STRING          -- 'IN_PROGRESS', 'QUARANTINED', 'READY_FOR_STG'
)
USING DELTA;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.fact_sales_order_item_stg (
    FactSalesOrderItemKey BIGINT GENERATED ALWAYS AS IDENTITY,
    
    -- Business / Degenerate Keys
    SalesOrder STRING NOT NULL,
    SalesOrderItem STRING NOT NULL,
    
    -- Dimension Foreign Keys
    CustomerKey BIGINT NOT NULL,
    ProductKey BIGINT NOT NULL,
    OrganizationKey BIGINT NOT NULL,
    CreationDateKey INT,
    SalesOrderDateKey INT,
    RequestedDeliveryDateKey INT,
    
    -- Status and Descriptive Attributes
    SalesOrderItemCategory STRING,
    DeliveryStatus STRING,
    SDProcessStatus STRING,
    BillingStatus STRING,
    TransactionCurrency STRING,
    
    -- Transactional Metrics
    RequestedQuantity DECIMAL(18, 3),
    ConfdDeliveredQuantity DECIMAL(18, 3),
    NetAmount DECIMAL(18, 2),
    TaxAmount DECIMAL(18, 2),
    CostAmount DECIMAL(18, 2),
    GrossWeight DECIMAL(18, 3),
    NetWeight DECIMAL(18, 3),
    ItemVolume DECIMAL(18, 3),
    
    -- Quality and Audit Flags
    IsDeleted BOOLEAN,
    DqWarnings ARRAY<STRING>,
    _ingestion_timestamp TIMESTAMP,
    
    -- Relational Constraints (Unity Catalog)
    CONSTRAINT pk_fact_sales PRIMARY KEY (FactSalesOrderItemKey),
    CONSTRAINT fk_fact_customer FOREIGN KEY (CustomerKey) REFERENCES workspace.silver.dim_customer_stg(CustomerKey),
    CONSTRAINT fk_fact_product FOREIGN KEY (ProductKey) REFERENCES workspace.silver.dim_product_stg(ProductKey),
    CONSTRAINT fk_fact_organization FOREIGN KEY (OrganizationKey) REFERENCES workspace.silver.dim_organization_stg(OrganizationKey)
)
CLUSTER BY (SalesOrder, CreationDateKey);